In [ ]:
import os
import time
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from torch.utils.tensorboard import SummaryWriter
import optuna  # Optunaのインポート

# GPUの確認
if torch.cuda.is_available():
    print("GPU is available")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available, using CPU")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# データフォルダ
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/Defectlabel4x4/DefectLabels_4x4_test1"

# 座標データのロード
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# エッジ情報を読み込む
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# 欠陥なしデータのペア
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectLabel_nodefect.npy")


# 初期のペアリスト
train_pairs = []
val_pairs = []
test_pairs = []

# 欠陥なしデータを追加
for _ in range(8):
 train_pairs.append(defect_free_pair)
val_pairs.append(defect_free_pair)
test_pairs.append(defect_free_pair)

# ペアのデータをカウントして確認
def count_pair_occurrences(pairs, target_pair):
    return pairs.count(target_pair)

train_count = count_pair_occurrences(train_pairs, defect_free_pair)
val_count = count_pair_occurrences(val_pairs, defect_free_pair)
test_count = count_pair_occurrences(test_pairs, defect_free_pair)

print(f"\nDefect-free data occurrences in train_pairs: {train_count} (expected: 8)")
print(f"Defect-free data occurrences in val_pairs: {val_count} (expected: 1)")
print(f"Defect-free data occurrences in test_pairs: {test_count} (expected: 1)")

if train_count == 8 and val_count == 1 and test_count == 1:
    print("\nAll defect-free data pairs are correctly added.")
else:
    print("\nThere is an issue with adding defect-free data pairs.")

# データとラベルのペアを取得する関数
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # 欠陥なしデータの場合
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"無効なファイル名の形式: {file_name}")
        return None

# データとラベルのペアを作成
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# 有効なペアのみ取得
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]

# ランダムにサンプリング
num_samples = 1296
random_indices = np.random.choice(len(valid_pairs), num_samples, replace=False)

# サンプルをトレーニングデータとして取得
train_pairs = [valid_pairs[i] for i in random_indices]

# 残りのデータを取得
remaining_pairs = [valid_pairs[i] for i in range(len(valid_pairs)) if i not in random_indices]

# 残りのデータを1:1でバリデーションとテストに分割
val_pairs, test_pairs = train_test_split(remaining_pairs, test_size=0.5, random_state=42)

# サンプル数の取得
num_train_samples = len(train_pairs)
num_val_samples = len(val_pairs)
num_test_samples = len(test_pairs)

# データ準備の関数
def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []

    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # データとラベルを読み込む
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]

        # 座標データと応力データを結合してノード特徴量を作成
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)

    # データを結合しテンソルに変換
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y

# 各データセットを準備
train_x, train_y = prepare_data(train_pairs)
val_x, val_y = prepare_data(val_pairs)
test_x, test_y = prepare_data(test_pairs)

np.save(f"/home/nishioka/GNN/4x4trainindata_confirm1/train_pairs_{timestamp}.npy", np.array(train_pairs))
np.save(f"/home/nishioka/GNN/4x4trainindata_confirm1/val_pairs_{timestamp}.npy", np.array(val_pairs))
np.save(f"/home/nishioka/GNN/4x4trainindata_confirm1/test_pairs_{timestamp}.npy", np.array(test_pairs))

# 保存後のデータセットのペアを表示
print(f"\nSaved train pairs: {len(train_pairs)} samples")
print(f"Saved validation pairs: {len(val_pairs)} samples")
print(f"Saved test pairs: {len(test_pairs)} samples")

# Dataオブジェクトにまとめる
train_data = Data(x=train_x, edge_index=edge_index, y=train_y)
val_data = Data(x=val_x, edge_index=edge_index, y=val_y)
test_data = Data(x=test_x, edge_index=edge_index, y=test_y)

# データローダーの作成
train_loader = DataLoader([train_data], batch_size=32, shuffle=True)
val_loader = DataLoader([val_data], batch_size=32, shuffle=False)
test_loader = DataLoader([test_data], batch_size=32, shuffle=False)

# Output sizes
print(f"\nDataset Sizes:")
print(f"{'Training data size:':<25} {len(train_pairs)} samples")
print(f"{'Validation data size:':<25} {len(val_pairs)} samples")
print(f"{'Test data size:':<25} {len(test_pairs)} samples")

# Detailed Data Sizes
train_sample_size = len(train_pairs) * 3654
val_sample_size = len(val_pairs) * 3654
test_sample_size = len(test_pairs) * 3654

print(f"\nDetailed Data Sizes:")
print(f"{'Training Data size:':<25} {train_data.x.size(0)} (Expected: {len(train_pairs)} * 3654 = {train_sample_size})")
print(f"{'Training Edge size:':<25} {train_data.edge_index.size(1)}")
print(f"{'Training Label size:':<25} {train_data.y.size(0)} (Expected: {len(train_pairs)} * 3654 = {train_sample_size})")

print(f"\n{'Validation Data size:':<25} {val_data.x.size(0)} (Expected: {len(val_pairs)} * 3654 = {val_sample_size})")
print(f"{'Validation Edge size:':<25} {val_data.edge_index.size(1)}")
print(f"{'Validation Label size:':<25} {val_data.y.size(0)} (Expected: {len(val_pairs)} * 3654 = {val_sample_size})")

print(f"\n{'Test Data size:':<25} {test_data.x.size(0)} (Expected: {len(test_pairs)} * 3654 = {test_sample_size})")
print(f"{'Test Edge size:':<25} {test_data.edge_index.size(1)}")
print(f"{'Test Label size:':<25} {test_data.y.size(0)} (Expected: {len(test_pairs)} * 3654 = {test_sample_size})")


# Display detailed data pairs for Training, Validation, and Test datasets
print("\n--- Training Data Sample ---")
for i, (data_file, label_file) in enumerate(train_pairs[:2], 1):  # Display first two samples
    print(f"Sample {i}: data: {data_file}, label: {label_file}")

print("\n--- Validation Data Sample ---")
for i, (data_file, label_file) in enumerate(val_pairs[:2], 1):
    print(f"Sample {i}: data: {data_file}, label: {label_file}")

print("\n--- Test Data Sample ---")
for i, (data_file, label_file) in enumerate(test_pairs[:2], 1):
    print(f"Sample {i}: data: {data_file}, label: {label_file}")

# Xavier初期化関数
def init_weights(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, GATConv):
        # GATConvの内部で使用されているlin_srcとlin_dstが存在し、かつそれらがNoneでない場合のみ初期化
        if hasattr(m, 'lin_src') and m.lin_src is not None and hasattr(m.lin_src, 'weight'):
            torch.nn.init.xavier_uniform_(m.lin_src.weight)
        if hasattr(m, 'lin_dst') and m.lin_dst is not None and hasattr(m.lin_dst, 'weight'):
            torch.nn.init.xavier_uniform_(m.lin_dst.weight)


# 7層のGATモデルの定義
class GATModel(torch.nn.Module):
    def __init__(self, hidden_channels=128, dropout_rate=0.2):
        super(GATModel, self).__init__()
        self.conv1 = GATConv(4, hidden_channels, heads=4)  # 出力次元: hidden_channels * 4
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=4)  # 出力次元: hidden_channels * 8
        self.conv3 = GATConv(hidden_channels * 8, hidden_channels * 4, heads=4)  # 出力次元: hidden_channels * 16
        self.conv4 = GATConv(hidden_channels * 16, hidden_channels, heads=8)  # 出力次元: hidden_channels * 4
        self.conv5 = GATConv(hidden_channels * 8, hidden_channels, heads=4)  # 出力次元: hidden_channels * 4
        self.conv6 = GATConv(hidden_channels * 4, hidden_channels, heads=4)  # 出力次元: hidden_channels * 4
        self.conv7 = GATConv(hidden_channels * 4, hidden_channels)  # headsデフォルト1、出力次元: hidden_channels
        self.fc_defect = torch.nn.Linear(hidden_channels, 1)  # 回帰タスク用
        self.dropout = torch.nn.Dropout(p=dropout_rate)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # 残差接続を追加
        x1 = F.relu(self.conv1(x, edge_index))
        x1 = self.dropout(x1)
        x2 = F.relu(self.conv2(x1, edge_index))
        x2 = self.dropout(x2) + x1  # 残差接続
        x3 = F.relu(self.conv3(x2, edge_index))
        x3 = self.dropout(x3) + x2  # 残差接続
        x4 = F.relu(self.conv4(x3, edge_index))
        x4 = self.dropout(x4) + x3  # 残差接続
        x5 = F.relu(self.conv5(x4, edge_index))
        x5 = self.dropout(x5) + x4  # 残差接続
        x6 = F.relu(self.conv6(x5, edge_index))
        x6 = self.dropout(x6) + x5  # 残差接続
        x7 = F.relu(self.conv7(x6, edge_index))
        x7 = self.fc_defect(x7)
        return torch.sigmoid(x7)

# ハイパーパラメータ設定
hidden_channels = 64
learning_rate = 0.0005
batch_size = 8
epochs = 1500
weight_decay = 5e-4
patience = 100  # Early Stopping用
dropout_rate = 0.2

# # Optunaのobjective関数の定義
# def objective(trial):
#     # ハイパーパラメータの提案
#     hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128, 256])
#     learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
#     weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
#     dropout_rate = trial.suggest_uniform('dropout_rate', 0.0, 0.5)
#     patience = trial.suggest_int('patience', 10, 100)
#     step_size = trial.suggest_int('step_size', 50, 200)
#     gamma = trial.suggest_uniform('gamma', 0.1, 0.9)


# モデル、損失関数、オプティマイザの定義
model = GATModel(hidden_channels=hidden_channels, dropout_rate=dropout_rate).to(device)
model.apply(init_weights)  # Xavier初期化
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
loss_fn = torch.nn.MSELoss()

# 学習率スケジューラの定義
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.1)

# TensorBoardの初期化
writer = SummaryWriter(log_dir=f'/home/nishioka/GNN/GNNlogs/{timestamp}')

# EarlyStopping クラス定義
class EarlyStopping:
    """Early stopping utility to stop training when validation loss doesn't improve."""
    def __init__(self, patience=patience, verbose=False, delta=0, path='/home/nishioka/GNN/checkpoint1.pt', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# EarlyStoppingクラスのインスタンス作成
early_stopping = EarlyStopping(patience=patience, verbose=True, path=f'/home/nishioka/GNN/GNNmodel/gat_model_best_{timestamp}.pth')

train_losses = []
val_losses = []
test_losses = []
start_time = time.time()

# 学習ループの開始
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))

    # 検証ステップ
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch)
            loss = loss_fn(out, batch.y.view(-1, 1))
            val_loss += loss.item()

    val_losses.append(val_loss / len(val_loader))

    # 学習率のステップ
    scheduler.step()

    # EarlyStoppingに現在のバリデーション損失とモデルを渡す
    early_stopping(val_loss / len(val_loader), model)

    # EarlyStoppingがTrueなら訓練を終了
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        stopped_epoch = epoch
        break
    else:
        stopped_epoch = epochs

    # TensorBoardに損失を記録
    writer.add_scalar('Loss/train', train_losses[-1], epoch)
    writer.add_scalar('Loss/val', val_losses[-1], epoch)

    # 1, 10, 30, 50, 100, 150, 200, 250... のタイミングで結果を出力
    if epoch == 1 or epoch == 10 or epoch == 30 or (epoch % 50 == 0):
        elapsed_time = time.time() - start_time
        print(f'Epoch {epoch}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Time: {elapsed_time:.2f}s')

# テストデータでの評価
model.load_state_dict(torch.load(f'/home/nishioka/GNN/GNNmodel/gat_model_best_{timestamp}.pth'))  # 最良モデルをロード

test_loss = 0
test_preds = []
test_targets = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        test_loss += loss.item()
        test_preds.append(out.cpu().numpy())
        test_targets.append(batch.y.cpu().numpy())

test_loss /= len(test_loader)
test_preds = np.concatenate(test_preds)
test_targets = np.concatenate(test_targets)

# RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(test_targets, test_preds))

# MAE (Mean Absolute Error)
mae = mean_absolute_error(test_targets, test_preds)

# R2 Score
r2 = r2_score(test_targets, test_preds)

print(f'Test Loss: {test_loss:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2 Score: {r2:.4f}')

# 学習時間の計測終了
elapsed_time = time.time() - start_time
print(f"Total training time: {elapsed_time:.2f} seconds")

# モデル、学習結果の保存
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gat_model_final_{timestamp}.pth')

# 損失とハイパーパラメータ情報をCSVに保存
loss_data = pd.DataFrame({
    'Epoch': range(1, stopped_epoch + 1),
    'Training Loss': train_losses[:stopped_epoch],
    'Validation Loss': val_losses[:stopped_epoch]
})
loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/loss_data_{timestamp}.csv', index=False)

# 損失をプロット
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss', color='blue')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', color='orange')
plt.yscale('log')  # 対数スケール
plt.xlabel('Epoch')
plt.ylabel('Loss (Log Scale)')
plt.title(f'Training and Validation Loss {type(model).__name__} - ({timestamp})')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/loss_plot_{type(model).__name__}_{timestamp}.png')
plt.show()

# パラメータ情報をテーブルとして保存
# テーブルデータの作成
table_data = [
    ["Model", type(model).__name__],
    ["Hidden Channels", hidden_channels],
    ["Learning Rate", learning_rate],
    ["Batch Size", batch_size],
    ["Epochs", epochs],
    ["Weight Decay", weight_decay],
    ["Patience", patience],
    ["Dropout Rate", dropout_rate],
    ["Training Samples", num_train_samples],
    ["Validation Samples", num_val_samples],
    ["Test Samples", num_test_samples]
]

# ハイパーパラメータのCSV保存
hyperparameter_data = pd.DataFrame(table_data, columns=["Parameter", "Value"])
hyperparameter_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/hyperparameters_{timestamp}.csv', index=False)

# 学習時間をファイルに保存
with open(f'/home/nishioka/GNN/GNNmodelcsv/training_time_{timestamp}.txt', 'w') as f:
    f.write(f"Total training time: {elapsed_time:.2f} seconds\n")

# テスト結果を保存
test_loss_data = pd.DataFrame({
    'Test Loss': [test_loss],
    'RMSE': [rmse],
    'MAE': [mae],
    'R2 Score': [r2]
})
test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/test_loss_{timestamp}.csv', index=False)

# モデルの保存（ファイル名にタイムスタンプを追加）
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_{timestamp}.pth')
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodelweights/{type(model).__name__}_weights_{timestamp}.pth')

# TensorBoardのWriterを閉じる
writer.close()

# 学習サマリの出力
print("\nTraining Summary:")
print(f"Total Epochs Run: {stopped_epoch}")
if early_stopping.early_stop:
    print(f"Early stopping at epoch {stopped_epoch}")
else:
    print("Training continued without early stopping.")
print(f"Final Training Loss: {train_losses[stopped_epoch-1]:.4f}")
print(f"Best Validation Loss: {early_stopping.val_loss_min:.4f}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"Total Training Time: {elapsed_time:.2f} seconds")

# データの保存
np.save(f'/home/nishioka/GNN/4x4trainindata_confirm1/train_pairs{type(model).__name__}_{timestamp}.npy', train_pairs)
np.save(f'/home/nishioka/GNN/4x4trainindata_confirm1/val_pairs{type(model).__name__}_{timestamp}.npy', val_pairs)
np.save(f'/home/nishioka/GNN/4x4trainindata_confirm1/test_pairs{type(model).__name__}_{timestamp}.npy', test_pairs)

print("Data and labels saved successfully.")
